In [1]:
import pandas as pd

df = pd.read_csv("insurance.csv")

print("Lignes :", len(df))
print("Colonnes :", len(df.columns))
print("Doublons :", df.duplicated().sum())

Lignes : 1338
Colonnes : 7
Doublons : 1


In [2]:
df = df.rename(columns={
    "age": "age",
    "sex": "genre",
    "bmi": "imc",
    "children": "enfants",
    "smoker": "fumeur",
    "region": "region",
    "charges": "charges"
})

df = df.drop_duplicates()

print("Lignes après nettoyage :", len(df))
print(df.columns.tolist())

Lignes après nettoyage : 1337
['age', 'genre', 'imc', 'enfants', 'fumeur', 'region', 'charges']


In [3]:
df['fumeur'] = df['fumeur'].map({'yes': 1, 'no': 0})
df['genre'] = df['genre'].map({'male': 1, 'female': 0})

print(df[['genre', 'fumeur']].head())

   genre  fumeur
0      0       1
1      1       0
2      1       0
3      1       0
4      1       0


In [4]:
df["region"] = df["region"].map({
    'northeast': 0,
    'northwest': 1,
    'southeast': 2,
    'southwest': 3
})

print(df['region'].head())

0    3
1    2
2    2
3    1
4    1
Name: region, dtype: int64


In [ ]:
# On définit X et Y
X = df.drop(columns=['charges'])
Y = df['charges']

print(X.shape)
print(Y.shape)

(1337, 6)
(1337,)


In [6]:
# Séparation des données en Train/Test

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("Entraînement :", len(X_train), "lignes")
print("Test :", len(X_test), "lignes")


Entraînement : 1069 lignes
Test : 268 lignes


In [8]:
# Statistiques des charges réelles
print("Moyenne :", round(df['charges'].mean(), 0), "$")
print("Minimum :", round(df['charges'].min(), 0), "$")
print("Maximum :", round(df['charges'].max(), 0), "$")

Moyenne : 13279.0 $
Minimum : 1122.0 $
Maximum : 63770.0 $


In [ ]:
# Moyennes par profil fumeur/genre
groupes = df.groupby(['fumeur', 'genre'])['charges'].mean().round(0)
print(groupes)


fumeur  genre
0       0         8762.0
        1         8100.0
1       0        30679.0
        1        33042.0
Name: charges, dtype: float64


In [13]:
# Test du troisième modèle : Random Forest
from sklearn.ensemble import RandomForestRegressor

# Créer le modèle
modele3 = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# Entraîner le modèle
modele3.fit(X_train, Y_train)

# Faire des prédictions sur les données de test
predictions3 = modele3.predict(X_test)

# Évaluer le modèle
r2_3 = r2_score(Y_test, predictions3)
mae_3 = mean_absolute_error(Y_test, predictions3)

print("Random Forest")
print("R² :", round(r2_3, 3))
print("MAE :", round(mae_3, 0), "$")

Random Forest
R² : 0.902
MAE : 2410.0 $


In [14]:
# Tableau comparatif des 3 modèles
import pandas as pd

resultats = pd.DataFrame({
    'Modèle': ['Régression Linéaire', 'Arbre de Décision', 'Random Forest'],
    'R²': [r2_1, r2_2, r2_3],
    'MAE ($)': [mae_1, mae_2, mae_3]
})

print(resultats)

                Modèle        R²      MAE ($)
0  Régression Linéaire  0.806847  4182.353155
1    Arbre de Décision  0.898111  2591.668249
2        Random Forest  0.902330  2410.237884


In [ ]:
# Prédiction pour une personne
# age=35, genre=homme(1), imc=28, enfants=2, fumeur=non(0), region=nord-est(0)
personne = [[35, 1, 28, 2, 0, 0]]

charge_predite = modele3.predict(personne)

print("Charge estimée :", round(charge_predite[0], 2), "$")

Charge estimée : 7060.47 $


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
